# Non-parametric QP with Dummy Normal Inputs

This notebook is converted from the parametric QP notebook. The original problem used parameter-dependent constraints

\[
Ay=b+Bx,\quad l+Lx\le y\le u+Ux.
\]

Here, the optimization problem is **non-parametric**:

\[
\begin{aligned}
\min_y \quad & \frac12 y^\top Qy + c^\top y \\
\text{s.t.}\quad & Ay=b,\\
& Cy\le d,\\
& l\le y\le u.
\end{aligned}
\]

Because the neural-network framework still expects an input, we keep a dummy input vector \(x_{\mathrm{dummy}}\in\mathbb{R}^5\). These dummy inputs are sampled from a standard normal distribution and are **not used inside the QP constraints or objective**.


In [20]:
import numpy as np
import pandas as pd
from nlpoptnet import NLPOptNet

## Configuration and model object

In [21]:
CONFIG = {
    'epochs': 100,
    'batch_size': 1,
    'learning_rate': 1e-3,
    'train_frac': 0.5,
    'hidden_size': 64,
    'hidden_layers': 2,
    'seed': 42,
    'alpha_consistency': 10.0,
    'cp_mode': 'fixed',
    'cp_iters': 300,
    'cp_tol': 1e-9,
    'safety': 0.95,
    'knorm_iters': 15,
    'knorm_seed': 42,
    'adjoint_iters': 20,
    'k_layer': 1,
    'use_ruiz': True,
    'ruiz_iters': 5,
    'dtype': 'float64',
    'print_every': 10,
    'device': 'auto',
    'verbose': True,
}

# The QP itself is non-parametric, but the NN receives dummy inputs.
p_dummy = 5
n_y = 10

model = NLPOptNet(config=CONFIG, type='qp', name='Example_QP_NonParametric_DummyInput')
x = model.add_parameter([f'x{i+1}' for i in range(p_dummy)])
y = model.add_variable([f'y{i+1}' for i in range(n_y)])

## Define non-parametric QP constants

Compared with the parametric notebook, `B`, `L`, and `U` are removed because the constraints no longer depend on the input.

In [22]:
Q = np.array([[2.1, 0.1, 0.0, 0.05, 0.0, 0.02, 0.0, 0.0, 0.0, 0.0],
              [0.1, 2.0, 0.08, 0.0, 0.03, 0.0, 0.02, 0.0, 0.0, 0.0],
              [0.0, 0.08, 1.9, 0.07, 0.0, 0.0, 0.0, 0.02, 0.0, 0.0],
              [0.05, 0.0, 0.07, 2.2, 0.06, 0.0, 0.0, 0.0, 0.02, 0.0],
              [0.0, 0.03, 0.0, 0.06, 2.05, 0.0, 0.0, 0.0, 0.0, 0.02],
              [0.02, 0.0, 0.0, 0.0, 0.0, 1.8, 0.05, 0.0, 0.0, 0.0],
              [0.0, 0.02, 0.0, 0.0, 0.0, 0.05, 1.85, 0.04, 0.0, 0.0],
              [0.0, 0.0, 0.02, 0.0, 0.0, 0.0, 0.04, 1.95, 0.03, 0.0],
              [0.0, 0.0, 0.0, 0.02, 0.0, 0.0, 0.0, 0.03, 1.75, 0.04],
              [0.0, 0.0, 0.0, 0.0, 0.02, 0.0, 0.0, 0.0, 0.04, 1.88]], dtype=float)

c = np.array([0.65, 0.75, 0.85, 0.95, 1.05, 0.55, 0.7, 0.9, 1.1, 1.2], dtype=float)

A = np.array([[1.0, 0.0, 0.0, 0.0, 0.0, 0.2, -0.1, 0.0, 0.05, 0.0],
              [0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.15, -0.05, 0.0, 0.1],
              [0.0, 0.0, 1.0, 0.0, 0.0, -0.1, 0.0, 0.1, 0.05, 0.0],
              [0.0, 0.0, 0.0, 1.0, 0.0, 0.05, 0.0, 0.0, -0.1, 0.15],
              [0.0, 0.0, 0.0, 0.0, 1.0, 0.0, -0.05, 0.1, 0.0, -0.1]], dtype=float)

b = np.array([0.28, -0.1915, 0.083, -0.0175, 0.1455], dtype=float)

C = np.array([[0.4, -0.2, 0.0, 0.1, 0.0, 0.0, 0.2, 0.0, 0.0, -0.1],
              [-0.1, 0.3, 0.2, 0.0, 0.0, 0.1, 0.0, -0.2, 0.0, 0.0],
              [0.0, 0.1, -0.3, 0.2, 0.1, 0.0, 0.0, 0.0, 0.2, 0.0],
              [0.2, 0.0, 0.0, -0.2, 0.3, -0.1, 0.0, 0.0, 0.0, 0.1],
              [0.0, -0.1, 0.1, 0.0, -0.2, 0.2, -0.1, 0.3, 0.0, 0.0]], dtype=float)

d = np.array([0.656, 0.521, 0.509, 0.644, 0.704], dtype=float)

l = np.array([-1.05, -1.4, -1.15, -1.3, -1.13, -0.95, -1.5, -1.07, -1.35, -1.2], dtype=float)
u = np.array([1.45, 1.1, 1.35, 1.2, 1.37, 1.55, 1.0, 1.43, 1.15, 1.3], dtype=float)

model.Q = model.matrix(Q)
model.c = model.vector(c)
model.A = model.matrix(A)
model.b = model.vector(b)
model.C = model.matrix(C)
model.d = model.vector(d)
model.l = model.vector(l)
model.u = model.vector(u)

## Objective and constraints

The dummy input `x` is intentionally absent from the QP definition.

In [23]:
model.objective(0.5 * model.quad(model.Q, y) + model.lin(model.c, y))

# Non-parametric constraints: no Bx, no Lx, no Ux.
model.constraints.equality.add(model.lin(model.A, y) == model.b)
model.constraints.inequality.add(model.lin(model.C, y) <= model.d)
model.constraints.box.add(var=y, lower=model.l, upper=model.u)

## Generate dummy normal inputs

These samples are only neural-network inputs. They do not change the feasible set or objective. Therefore, the true optimizer of the QP is the same for every dummy input. The network learns to map arbitrary dummy inputs to the same projected/optimal solution.

In [24]:
rng = np.random.default_rng(CONFIG['seed'])
num_samples = 2

# Standard normal dummy inputs. You can change loc/scale if needed.
X_dummy = rng.normal(loc=0.0, scale=1.0, size=(num_samples, p_dummy)).astype(float)

df_dummy = pd.DataFrame(X_dummy, columns=[f'x{i+1}' for i in range(p_dummy)])
df_dummy.to_csv('dummy_normal_inputs.csv', index=False)

In [25]:
model.dataset(parameters="dummy_normal_inputs.csv")

## Build, train, and use the model

In [26]:
model.build()

🤖 Model build successfully!


In [27]:
result = model.optimize()
run_dir = result['output_dir']
print('run_dir =', run_dir)

▶️ Model training started! [2026-04-30 11:44:26]

       Epoch |                      Training                       |                     Validation                     
------------------------------------------------------------------------------------------------------------------------
             |         Loss          Obj           Eq         Ineq |         Loss          Obj           Eq         Ineq
     001/100 |   1.5246e+01  -5.9818e-01   1.8399e-10   0.0000e+00 |   1.7860e+01  -5.9717e-01   4.5474e-10   0.0000e+00
     010/100 |   1.4405e+00  -5.9910e-01   3.7691e-10   0.0000e+00 |   1.3712e+01  -5.9730e-01   4.0635e-10   0.0000e+00
     020/100 |  -2.1748e-02  -5.9908e-01   2.7428e-10   0.0000e+00 |   1.1065e+01  -5.9734e-01   3.8709e-10   0.0000e+00
     030/100 |  -2.2215e-01  -5.9917e-01   1.0652e-10   0.0000e+00 |   1.1653e+01  -5.9735e-01   4.1774e-10   0.0000e+00
     040/100 |  -5.4373e-01  -5.9917e-01   3.3639e-10   0.0000e+00 |   1.2598e+01  -5.9736e-01   4.2081

In [ ]:
model.summary()

In [ ]:
model.plot_history()

## Prediction on new dummy normal inputs

In [ ]:
X_test = rng.normal(loc=0.0, scale=1.0, size=(3, p_dummy)).astype(float)

pred = model.predict(X_test, projection_backend='jax')
pred_arr = np.asarray(pred)
print('predicted y shape:', pred_arr.shape)
pd.DataFrame(pred_arr, columns=[f'y{i+1}' for i in range(n_y)])

## Optional: verify that all predictions satisfy the same non-parametric constraints

In [ ]:
eq_residual = pred_arr @ A.T - b
ineq_violation = np.maximum(pred_arr @ C.T - d, 0.0)
lower_violation = np.maximum(l - pred_arr, 0.0)
upper_violation = np.maximum(pred_arr - u, 0.0)

checks = pd.DataFrame({
    'eq_l2': np.linalg.norm(eq_residual, axis=1),
    'ineq_max_violation': np.max(ineq_violation, axis=1),
    'box_lower_max_violation': np.max(lower_violation, axis=1),
    'box_upper_max_violation': np.max(upper_violation, axis=1),
})
checks